# SNF Mining Manual XLS To DuckDB

This notebook standardizes the manually exported S&P Global mining `.xls` file into a small DuckDB database. It is intended as a transitional ingestion path for the manual export and will likely be deprecated once the richer scrape becomes the primary source.

**Purpose**
- read the manual `.xls` export
- normalize core fields into stable tables
- preserve long-form work history text for audit and later reprocessing
- write the cleaned result to DuckDB

**Primary output**
- `data/snf_mining/processed/stage_0/manual_xls/snf_mining_manual_export.duckdb`

**Tables written**
- `source_files`: provenance for the imported file
- `properties`: one cleaned row per property
- `property_texts`: raw narrative text fields such as `full_work_history`
- `property_work_history_events`: derived event rows parsed from parenthetical date snippets
- `raw_property_records`: original row payloads stored as JSON for traceability


## Dependencies

This notebook expects:
- `pandas`
- `duckdb`
- `python-calamine` or `xlrd` for legacy `.xls` reading

Run this notebook before the OpenAI enrichment notebook if you want the enrichment workflow to load standardized manual-export tables from DuckDB.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Locate the repository root by walking upward until the expected folders appear."""
    candidates = [start or Path.cwd(), *(start or Path.cwd()).parents]
    for candidate in candidates:
        if (candidate / "gnt").exists() and (candidate / "orchestration").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
SOURCE_PATH = PROJECT_ROOT / "data" / "snf_mining" / "raw" / "SPGlobal_Export_3-31-2026_d895a56a-e8e3-4698-a5c7-b2931645b812.xls"
OUTPUT_DIR = PROJECT_ROOT / "data" / "snf_mining" / "processed" / "stage_0" / "manual_xls"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = OUTPUT_DIR / "snf_mining_manual_export.duckdb"

SOURCE_PATH, DB_PATH


In [ ]:
COLUMN_MAP = {
    "PROP_NAME": "property_name",
    "PROP_ID": "property_id",
    "PRIMARY_COMMODITY": "primary_commodity",
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude",
    "START_UP_YR": "actual_start_up_year",
    "ACTUAL_CLOSURE_YR": "actual_closure_year",
    "FULL_WORK_HISTORY": "full_work_history_raw",
}

EXPECTED_MACHINE_COLUMNS = list(COLUMN_MAP)
WORK_HISTORY_PATTERN = re.compile(r"\(([^\)]+)\)\s*([^\(]+?)(?=\s*\([^\)]+\)|$)")


def normalize_string(value: object) -> str | None:
    # Normalize whitespace and the platform's placeholder `NA` values.
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.upper() == "NA":
        return None
    return re.sub(r"\s+", " ", text)


def parse_spglobal_number(value: object) -> float | None:
    # S&P exports use decimal commas and sometimes accounting parentheses.
    text = normalize_string(value)
    if text is None:
        return None

    is_negative = text.startswith("(") and text.endswith(")")
    text = text.strip("()")
    text = text.replace(".", "") if text.count(",") == 1 and text.count(".") > 1 else text
    text = text.replace(",", ".")

    try:
        number = float(text)
    except ValueError:
        return None

    return -number if is_negative else number


def parse_year(value: object) -> int | None:
    # Some year fields contain text; keep the first four-digit year when present.
    text = normalize_string(value)
    if text is None:
        return None
    match = re.search(r"\d{4}", text)
    return int(match.group(0)) if match else None


def locate_machine_header(raw_sheet: pd.DataFrame) -> int:
    # The export starts with presentation rows, so we scan for the machine header.
    for idx in raw_sheet.index:
        values = [normalize_string(v) for v in raw_sheet.loc[idx].tolist()]
        if values[: len(EXPECTED_MACHINE_COLUMNS)] == EXPECTED_MACHINE_COLUMNS:
            return int(idx)
    raise ValueError("Could not locate the machine-readable header row in the export.")


def read_manual_export(path: Path) -> tuple[pd.DataFrame, dict[str, str]]:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        raw_sheet = pd.read_csv(path, header=None, dtype=str, keep_default_na=False)
    elif suffix == ".xls":
        # Prefer calamine for speed, but keep xlrd as a fallback for older setups.
        try:
            raw_sheet = pd.read_excel(path, sheet_name=0, header=None, dtype=str, engine="calamine")
        except Exception:
            raw_sheet = pd.read_excel(path, sheet_name=0, header=None, dtype=str, engine="xlrd")
    else:
        raise ValueError(f"Unsupported source format: {path.suffix}")

    machine_header_idx = locate_machine_header(raw_sheet)
    human_header_idx = machine_header_idx - 1

    machine_headers = [normalize_string(v) for v in raw_sheet.loc[machine_header_idx].tolist()]
    human_headers = [normalize_string(v) for v in raw_sheet.loc[human_header_idx].tolist()]
    header_pairs = {
        machine: human
        for machine, human in zip(machine_headers, human_headers)
        if machine in COLUMN_MAP
    }

    data = raw_sheet.loc[machine_header_idx + 1 :, : len(machine_headers) - 1].copy()
    data.columns = machine_headers
    data = data[EXPECTED_MACHINE_COLUMNS].copy()

    # Drop visual spacer rows without paying the cost of row-wise Python calls.
    stripped = data.fillna("").astype(str).apply(lambda column: column.str.strip())
    informative = stripped.ne("") & ~stripped.apply(lambda column: column.str.upper().eq("NA"))
    keep_mask = informative.any(axis=1)
    data = data.loc[keep_mask].reset_index(drop=True)
    return data, header_pairs


def build_property_tables(raw_records: pd.DataFrame, source_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    standardized = raw_records.rename(columns=COLUMN_MAP).copy()

    for column in ["property_name", "property_id", "primary_commodity", "full_work_history_raw"]:
        standardized[column] = standardized[column].map(normalize_string)

    standardized["latitude"] = standardized["latitude"].map(parse_spglobal_number)
    standardized["longitude"] = standardized["longitude"].map(parse_spglobal_number)
    standardized["actual_start_up_year"] = standardized["actual_start_up_year"].map(parse_year)
    standardized["actual_closure_year"] = standardized["actual_closure_year"].map(parse_year)

    standardized.insert(0, "source_file", source_path.name)
    standardized.insert(1, "source_format", source_path.suffix.lower().lstrip("."))
    standardized.insert(2, "source_property_key", standardized["property_id"].map(lambda value: f"spglobal_manual:{value}" if value else None))
    standardized.insert(3, "source_row_number", standardized.index + 1)

    properties = standardized[[
        "source_file",
        "source_format",
        "source_row_number",
        "source_property_key",
        "property_id",
        "property_name",
        "primary_commodity",
        "latitude",
        "longitude",
        "actual_start_up_year",
        "actual_closure_year",
    ]].copy()

    property_texts = standardized.loc[
        standardized["full_work_history_raw"].notna(),
        ["property_id", "source_property_key", "full_work_history_raw"],
    ].copy()
    property_texts = property_texts.rename(columns={"full_work_history_raw": "raw_text"})
    property_texts.insert(2, "field_name", "full_work_history")

    # Keep the raw narrative text, but also derive a simple event table from
    # repeated `(date) text` segments for downstream relational use.
    work_history_events: list[dict[str, object]] = []
    for row in standardized.loc[standardized["full_work_history_raw"].notna()].itertuples(index=False):
        matches = list(WORK_HISTORY_PATTERN.finditer(row.full_work_history_raw))
        if not matches:
            work_history_events.append({
                "property_id": row.property_id,
                "source_property_key": row.source_property_key,
                "event_sequence": 1,
                "event_date_raw": None,
                "event_date": None,
                "event_text": row.full_work_history_raw,
                "parse_strategy": "unparsed_full_text",
            })
            continue

        for event_sequence, match in enumerate(matches, start=1):
            event_date_raw = normalize_string(match.group(1))
            event_text = normalize_string(match.group(2))
            event_date = pd.to_datetime(event_date_raw, errors="coerce")
            work_history_events.append({
                "property_id": row.property_id,
                "source_property_key": row.source_property_key,
                "event_sequence": event_sequence,
                "event_date_raw": event_date_raw,
                "event_date": event_date.date().isoformat() if pd.notna(event_date) else None,
                "event_text": event_text,
                "parse_strategy": "dated_parenthetical_segments",
            })

    property_work_history_events = pd.DataFrame(
        work_history_events,
        columns=[
            "property_id",
            "source_property_key",
            "event_sequence",
            "event_date_raw",
            "event_date",
            "event_text",
            "parse_strategy",
        ],
    )

    raw_payloads = [
        json.dumps({key: normalize_string(value) for key, value in record.items()}, ensure_ascii=True, sort_keys=True)
        for record in raw_records.to_dict(orient="records")
    ]

    raw_property_records = pd.DataFrame({
        "property_id": standardized["property_id"],
        "source_property_key": standardized["source_property_key"],
        "source_file": source_path.name,
        "raw_record_json": raw_payloads,
    })

    return properties, property_texts, property_work_history_events, raw_property_records


In [ ]:
raw_records, header_pairs = read_manual_export(SOURCE_PATH)
raw_records.head()


In [ ]:
properties, property_texts, property_work_history_events, raw_property_records = build_property_tables(
    raw_records=raw_records,
    source_path=SOURCE_PATH,
)

properties.head()


## Write DuckDB

The manual export is written as a small relational database. This database is the output of this notebook and the input expected by the OpenAI enrichment notebook.


In [ ]:
with duckdb.connect(DB_PATH) as con:
    con.execute(
        """
        CREATE OR REPLACE TABLE source_files (
            source_file VARCHAR,
            source_format VARCHAR,
            source_path VARCHAR,
            row_count BIGINT
        )
        """
    )
    con.execute(
        "INSERT INTO source_files VALUES (?, ?, ?, ?)",
        [SOURCE_PATH.name, SOURCE_PATH.suffix.lower().lstrip("."), str(SOURCE_PATH), len(raw_records)],
    )

    con.register("properties_df", properties)
    con.register("property_texts_df", property_texts)
    con.register("property_work_history_events_df", property_work_history_events)
    con.register("raw_property_records_df", raw_property_records)

    con.execute("CREATE OR REPLACE TABLE properties AS SELECT * FROM properties_df")
    con.execute("CREATE OR REPLACE TABLE property_texts AS SELECT * FROM property_texts_df")
    con.execute("CREATE OR REPLACE TABLE property_work_history_events AS SELECT * FROM property_work_history_events_df")
    con.execute("CREATE OR REPLACE TABLE raw_property_records AS SELECT * FROM raw_property_records_df")

    summary = {
        "properties": con.execute("SELECT COUNT(*) FROM properties").fetchone()[0],
        "property_texts": con.execute("SELECT COUNT(*) FROM property_texts").fetchone()[0],
        "property_work_history_events": con.execute("SELECT COUNT(*) FROM property_work_history_events").fetchone()[0],
        "raw_property_records": con.execute("SELECT COUNT(*) FROM raw_property_records").fetchone()[0],
    }

summary


In [ ]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    preview = con.execute(
        """
        SELECT
            property_id,
            property_name,
            primary_commodity,
            actual_start_up_year,
            actual_closure_year
        FROM properties
        ORDER BY property_id
        LIMIT 20
        """
    ).df()

preview


## Outputs

After a successful run, the manual-import artifact lives at:
- `data/snf_mining/processed/stage_0/manual_xls/snf_mining_manual_export.duckdb`

This notebook is intentionally limited to deterministic cleaning and packaging of the manual export. Model-based year imputation is handled separately in `snf_mining_openai_enrichment.ipynb`.
